In [0]:
%pip install xgboost scikit-learn pandas

In [0]:
import sys
import importlib
import random
import numpy as np
import pandas as pd

SRC_PATH = "/Workspace/Users/ariamostajeran99@gmail.com/stock_project_V2/stock-mlops-databricks/src"

if SRC_PATH not in sys.path:
    sys.path.append(SRC_PATH)

import config
import universe
import model_trainer
import dataset_builder
import evaluator
import signal_generator
import portfolio_backtester
import reporting_monitoring

importlib.reload(config)
importlib.reload(universe)
importlib.reload(model_trainer)
importlib.reload(dataset_builder)
importlib.reload(evaluator)
importlib.reload(signal_generator)
importlib.reload(portfolio_backtester)
importlib.reload(reporting_monitoring)

In [0]:
from config import (
    ML_DATASET_TABLE_NAME,
    GLOBAL_RANDOM_SEED,
    TOP_K_FEATURES,
    ROLLING_TRAIN_YEARS,
    ROLLING_TEST_MONTHS,
    CONFIDENCE_THRESHOLDS,
    TARGET_HORIZON_DAYS,
)

from universe import TRAIN_UNIVERSE
from model_trainer import ModelTrainer
from dataset_builder import DatasetBuilder
from evaluator import ModelEvaluator
from signal_generator import SignalGenerator
from portfolio_backtester import PortfolioBacktester
from reporting_monitoring import ReportingMonitoring

In [0]:
random.seed(GLOBAL_RANDOM_SEED)
np.random.seed(GLOBAL_RANDOM_SEED)

ml_df = spark.table(ML_DATASET_TABLE_NAME).toPandas()
ml_df["Date"] = pd.to_datetime(ml_df["Date"], errors="coerce")

print("ML dataset rows:", len(ml_df))
print("Tickers in ML dataset:", sorted(ml_df["Ticker"].dropna().unique().tolist()))
print(ml_df["Ticker"].value_counts())

In [0]:
builder = DatasetBuilder(
    target_horizon_days=TARGET_HORIZON_DAYS
)

trainer = ModelTrainer(random_state=GLOBAL_RANDOM_SEED)
evalr = ModelEvaluator(confidence_thresholds=CONFIDENCE_THRESHOLDS)

NEWS_DATE_LIMITS = {
    "AAPL": ("2016-02-19", "2026-12-31"),
    "TSLA": ("2020-01-02", "2026-12-31"),
    "MSFT": ("2020-01-02", "2026-12-31"),
}

In [0]:
tickers = [t for t in TRAIN_UNIVERSE if t in sorted(ml_df["Ticker"].dropna().unique().tolist())]
print("Training tickers:", tickers)

all_metrics = []
all_baseline_compare = []
all_predictions = []
all_feature_importance = []
all_probability_summary = []
all_feature_selection = []
all_best_params = []

for ticker in tickers:
    print(f"\nTraining model for {ticker} ...")

    ticker_feature_cols = builder.get_feature_columns(ml_df, ticker=ticker)

    ticker_df, model_feature_cols = trainer.prepare_single_ticker_dataset(
        ml_df=ml_df,
        feature_cols=ticker_feature_cols,
        ticker=ticker
    )

    ticker_df = ticker_df.dropna(
        subset=model_feature_cols + ["target_up_down", "naive_prediction"]
    ).reset_index(drop=True)

    ticker_df["Date"] = pd.to_datetime(ticker_df["Date"], errors="coerce")

    if ticker in NEWS_DATE_LIMITS:
        start_date, end_date = NEWS_DATE_LIMITS[ticker]
        ticker_df = ticker_df[
            (ticker_df["Date"] >= pd.Timestamp(start_date)) &
            (ticker_df["Date"] <= pd.Timestamp(end_date))
        ].copy().reset_index(drop=True)

    print(f"{ticker} rows after restriction:", len(ticker_df))
    print(f"{ticker} feature count:", len(model_feature_cols))

    # shorter rolling horizon for shorter-news-history assets
    train_years_for_ticker = 8 if ticker == "AAPL" else 3

    splits = trainer.get_walk_forward_splits(
        ticker_df,
        train_years=train_years_for_ticker,
        test_months=ROLLING_TEST_MONTHS
    )

    if len(splits) == 0:
        print(f"Skipping {ticker}: no valid rolling splits")
        continue

    for split_id, (train_df, test_df) in enumerate(splits, start=1):
        X_train_full = train_df[model_feature_cols].copy()
        y_train = train_df["target_up_down"].copy()

        X_test_full = test_df[model_feature_cols].copy()
        y_test = test_df["target_up_down"].copy()

        # pass 1: feature selection
        model_full = trainer.fit_xgboost(X_train_full, y_train)
        top_features = trainer.select_top_features(model_full, model_feature_cols, TOP_K_FEATURES)

        all_feature_selection.append({
            "Ticker": ticker,
            "split_id": split_id,
            "selected_feature_count": len(top_features),
            "selected_features": ", ".join(top_features)
        })

        X_train = X_train_full[top_features].copy()
        X_test = X_test_full[top_features].copy()

        # tuned model
        model, best_params, best_cv_score = trainer.tune_xgboost(X_train, y_train, n_splits=3)

        all_best_params.append({
            "Ticker": ticker,
            "split_id": split_id,
            "best_cv_auc": best_cv_score,
            **best_params
        })

        train_pred = model.predict(X_train)
        train_prob = model.predict_proba(X_train)[:, 1]

        test_pred = model.predict(X_test)
        test_prob = model.predict_proba(X_test)[:, 1]

        train_metrics = trainer.evaluate_predictions(y_train, train_pred, train_prob)
        test_metrics = trainer.evaluate_predictions(y_test, test_pred, test_prob)

        all_metrics.append({
            "Ticker": ticker,
            "model_name": f"p2_xgboost_{ticker}_binary_v2",
            "split_id": split_id,
            "dataset_split": "train",
            **train_metrics
        })

        all_metrics.append({
            "Ticker": ticker,
            "model_name": f"p2_xgboost_{ticker}_binary_v2",
            "split_id": split_id,
            "dataset_split": "test",
            **test_metrics
        })

        prediction_output_df = trainer.build_prediction_output(
            test_df,
            y_pred=test_pred,
            y_prob=test_prob,
            model_name=f"p2_xgboost_{ticker}_binary_v2"
        )
        prediction_output_df["dataset_split"] = "test"
        prediction_output_df["split_id"] = split_id

        all_predictions.append(prediction_output_df)

        naive_acc = (test_df["naive_prediction"] == test_df["target_up_down"]).mean()
        xgb_acc = (test_pred == y_test).mean()

        all_baseline_compare.append({
            "Ticker": ticker,
            "model_name": f"p2_xgboost_{ticker}_binary_v2",
            "split_id": split_id,
            "naive_test_accuracy": naive_acc,
            "xgboost_test_accuracy": xgb_acc,
            "accuracy_lift": xgb_acc - naive_acc,
            "mean_strategy_return": prediction_output_df["strategy_return"].mean(),
            "hit_rate": (prediction_output_df["strategy_return"] > 0).mean(),
        })

        importance_df = trainer.feature_importance_table(
            model=model,
            feature_cols=top_features,
            ticker=ticker
        )
        importance_df["split_id"] = split_id
        all_feature_importance.append(importance_df)

        test_conf = pd.Series(test_prob)
        all_probability_summary.append({
            "Ticker": ticker,
            "split_id": split_id,
            "test_prob_q25": float(test_conf.quantile(0.25)),
            "test_prob_median": float(test_conf.median()),
            "test_prob_q75": float(test_conf.quantile(0.75)),
            "test_prob_max": float(test_conf.max()),
            "train_start_date": train_df["Date"].min(),
            "train_end_date": train_df["Date"].max(),
            "test_start_date": test_df["Date"].min(),
            "test_end_date": test_df["Date"].max(),
            "train_rows": len(train_df),
            "test_rows": len(test_df),
        })

In [0]:
metrics_df = pd.DataFrame(all_metrics)
baseline_compare_df = pd.DataFrame(all_baseline_compare)
all_predictions_df = pd.concat(all_predictions, axis=0, ignore_index=True) if all_predictions else pd.DataFrame()
importance_df = pd.concat(all_feature_importance, axis=0, ignore_index=True) if all_feature_importance else pd.DataFrame()
probability_summary_df = pd.DataFrame(all_probability_summary)
feature_selection_df = pd.DataFrame(all_feature_selection)
best_params_df = pd.DataFrame(all_best_params)

display(metrics_df)
display(baseline_compare_df.sort_values("accuracy_lift", ascending=False))
display(best_params_df)

In [0]:
eval_by_stock_df = evalr.evaluate_by_stock(all_predictions_df)
eval_by_confidence_df = evalr.evaluate_confidence_thresholds(all_predictions_df)
eval_confidence_by_stock_df = evalr.evaluate_confidence_by_stock(all_predictions_df)
eval_by_time_df = evalr.evaluate_time_segments(all_predictions_df, n_segments=10)

display(eval_by_stock_df)
display(eval_by_confidence_df)
display(eval_confidence_by_stock_df)
display(eval_by_time_df)

In [0]:
# choose per-stock thresholds manually from current research
up_thresholds = {
    "AAPL": 0.60,
    "MSFT": 0.60,
    "TSLA": 0.70,
}

down_thresholds = {
    "AAPL": 0.40,
    "MSFT": 0.40,
    "TSLA": 0.30,
}

generator = SignalGenerator(
    up_thresholds=up_thresholds,
    down_thresholds=down_thresholds,
    default_up_threshold=0.60,
    default_down_threshold=0.40
)

signals_df = generator.generate_signals(all_predictions_df)
signal_summary_df = generator.summarize_signals(signals_df)
latest_signals_df = generator.latest_signals(signals_df)

display(signal_summary_df)
display(latest_signals_df)